In [ ]:
from collections import Counter
import json
from pathlib import Path

import networkx as nx
import pandas as pd

In [ ]:
# GraphML file generated by Gephi
INPUT_GRAPHML = "../graphs/bipartite_podcasts_layout_threshold=1.graphml"

OUTPUT_JSON_D3 = "../d3/dataset.json"
OUTPUT_JSON_SIGMA = "../sigmajs-fancy/public/dataset.json"

NODE_SCALING = 2

BOUND = 800
BOUNDING_BOX = {"x": [-BOUND, BOUND], "y": [-BOUND, BOUND]}

LABEL_THRESHOLD = 12

CATEGORY_COLORS = {
  "Intellectual": "#00acc6",
  "Celebrity": "#018700",
  "Podcast": "#8c3bff",
  "Athlete": "#e6a500",
  "Politician": "#ff7ed1",
  "Other": "#6b004f",
  "Influencer": "#573b00",
  "Comedian": "#005659",
  "Political Pundit": "#15e18c"
}

## 1. Load laid out GraphML file and convert to Pandas DataFrames for nodes and edges

In [ ]:
G = nx.read_graphml(path=INPUT_GRAPHML)

_nodes_df = pd.DataFrame.from_dict(
    dict(G.nodes(data=True)), orient="index"
).reset_index(drop=True)
_edges_df = nx.to_pandas_edgelist(G=G)

nodes_df = _nodes_df[["x", "y", "label", "size", "category", "views"]].copy()
edges_df = _edges_df[["source", "target"]].copy()

label_to_index = {
    t[0]: i
    for i, t in enumerate(
        Counter(list(edges_df["source"]) + list(edges_df["target"])).most_common()
    )
}

nodes_df["key"] = nodes_df["label"].map(label_to_index)
nodes_df["size"] /= NODE_SCALING
nodes = nodes_df.to_dict(orient="records")

edges = [
    [label_to_index[e["source"]], label_to_index[e["target"]]]
    for e in edges_df.to_dict(orient="records")
]

## 2. Write data to JSON, used for d3 visualization

In [ ]:
clusters = [k for k, _ in Counter(nodes_df["category"]).most_common()]

data = {
    "nodes": nodes,
    "edges": edges,
    "clusters": [
        {"key": c, "clusterLabel": c, "color": CATEGORY_COLORS[c]}
        for c in clusters
    ],
    "bbox": BOUNDING_BOX,
    "labelThreshold": LABEL_THRESHOLD,
}

with open(OUTPUT_JSON_D3, "w") as f:
    json.dump(obj=data, fp=f, separators=(",", ":"))

## 3. Write data to JSON, used for sigma.js visualization (adds a few additional fields)

In [ ]:
data_sigma = {
    **data,
    "title": "Podcast guest appearance network around Donald Trump",
    "clusterTitle": "Categories",
    "description": """<p>
    This interactive visualization represents a <i>network</i> of podcast guest appearances around Donald Trump in the lead-up to the 2024 presidential election. Each
    <i>node</i> represents a podcast or podcast guest, and <i>edges</i> between nodes indicate that a guest appeared on a podcast
  </p>
  <p>
    This kind of visualization allows us to better understand the ecosystem of the YouTube podcaster community.
  </p>
  <p>
    This web application was developed by
    <a target="_blank" rel="noreferrer" href="https://tristanl.ee/">
      Tristan Lee
    </a>
    for a demo at
    <a target="_blank" rel="noreferrer" href="https://www.ire.org/training/conferences/nicar-2026/">
    NICAR 2026
    </a> using
    <a target="_blank" rel="noreferrer" href="https://gephi.org/">
      Gephi
    </a>,
    <a target="_blank" rel="noreferrer" href="https://reactjs.org/">
      react
    </a>,
    and
    <a target="_blank" rel="noreferrer" href="https://www.sigmajs.org">
      sigma.js
    </a>
    . You can read the source code
    <a target="_blank" rel="noreferrer" href="https://github.com/trislee/network-demo/">
      on GitHub
    </a>
    .
  </p>
  <p>
    The area of each node is related to the number of views the podcast or guest has across their videos. Nodes are colored based on the podcaster or guest category.
  </p>""",
}

with open(OUTPUT_JSON_SIGMA, "w") as f:
    json.dump(obj=data_sigma, fp=f, separators=(",", ":"))
